# AutoGluon Multiclass Classification - Fully Automated Pipeline

Implementasi AutoGluon untuk klasifikasi multiclass dengan transparansi penuh pada pipeline yang dibuat secara otomatis.

Dataset: 
- 777 training images (kaos & hoodie dengan 5 warna)
- 334 test images
- Labels: jenis (0=Kaos, 1=Hoodie), warna (0=merah, 1=kuning, 2=biru, 3=hitam, 4=putih)

## 1. Install Dependencies

In [ ]:
!pip install -q autogluon.multimodal

## 2. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from autogluon.multimodal import MultiModalPredictor
from sklearn.model_selection import train_test_split

print("Libraries imported successfully")

## 3. Load and Prepare Data

In [ ]:
# Load training data
train_df = pd.read_csv('train.csv')

# Add image paths
def get_image_path(img_id, base_dir='train/train'):
    jpg_path = os.path.join(base_dir, f"{img_id}.jpg")
    png_path = os.path.join(base_dir, f"{img_id}.png")
    return jpg_path if os.path.exists(jpg_path) else png_path

train_df['image_path'] = train_df['id'].apply(get_image_path)

print(f"Total training samples: {len(train_df)}")
print(f"\nDataset preview:")
print(train_df.head())
print(f"\nClass distribution:")
print(f"Jenis - Kaos: {(train_df['jenis']==0).sum()}, Hoodie: {(train_df['jenis']==1).sum()}")
print(f"Warna - Merah: {(train_df['warna']==0).sum()}, Kuning: {(train_df['warna']==1).sum()}, Biru: {(train_df['warna']==2).sum()}, Hitam: {(train_df['warna']==3).sum()}, Putih: {(train_df['warna']==4).sum()}")

## 4. Train AutoGluon Model for JENIS (Type) - Fully Automated

AutoGluon akan secara otomatis:
- Memilih model terbaik dari berbagai arsitektur
- Melakukan hyperparameter tuning
- Membuat ensemble dari multiple models
- Mengoptimalkan training pipeline

In [ ]:
# Prepare data for jenis classification
df_jenis = train_df[['image_path', 'jenis']].copy()
df_jenis.columns = ['image', 'label']

# Split train/validation
train_jenis, val_jenis = train_test_split(
    df_jenis, test_size=0.2, random_state=42, stratify=df_jenis['label']
)

print(f"Training samples: {len(train_jenis)}")
print(f"Validation samples: {len(val_jenis)}")

# Initialize AutoGluon predictor - minimal configuration
predictor_jenis = MultiModalPredictor(
    label='label',
    path='automl_jenis_model'
)

# Let AutoGluon decide everything
print("\nTraining JENIS model - AutoGluon will automatically select best pipeline...")
predictor_jenis.fit(
    train_data=train_jenis,
    time_limit=600  # 10 minutes - increase for better results
)

print("\nJENIS model training completed")

## 5. Inspect JENIS Model Pipeline (Transparency)

In [ ]:
# Inspect what AutoGluon decided to use
print("=" * 70)
print("JENIS MODEL PIPELINE TRANSPARENCY")
print("=" * 70)

# Get model information
print("\n1. MODEL ARCHITECTURE:")
print(f"   Model path: {predictor_jenis.path}")
print(f"   Problem type: {predictor_jenis.problem_type}")
print(f"   Label column: {predictor_jenis.label}")

# Get training configuration
try:
    print("\n2. TRAINING CONFIGURATION:")
    config = predictor_jenis._config
    if hasattr(config, 'model'):
        print(f"   Base model: {config.model}")
    if hasattr(config, 'optimization'):
        print(f"   Optimization: {config.optimization}")
except:
    print("   Configuration details stored internally")

# Check what's inside the model directory
print("\n3. SAVED ARTIFACTS:")
model_files = os.listdir(predictor_jenis.path)
for f in sorted(model_files)[:10]:
    print(f"   - {f}")

# Try to get model summary
print("\n4. MODEL SUMMARY:")
try:
    info = predictor_jenis.info()
    for key, value in info.items():
        print(f"   {key}: {value}")
except:
    print("   Model info available after predictions")

print("\n" + "=" * 70)

## 6. Evaluate JENIS Model

In [ ]:
# Evaluate on validation set
val_score = predictor_jenis.evaluate(val_jenis)
print("JENIS Model Performance:")
print(f"Validation Accuracy: {val_score['accuracy']:.4f}")

# Get predictions
predictions_jenis = predictor_jenis.predict(val_jenis)
print(f"\nSample predictions: {predictions_jenis[:10].tolist()}")
print(f"Sample true labels: {val_jenis['label'].values[:10].tolist()}")

## 7. Train AutoGluon Model for WARNA (Color) - Fully Automated

In [ ]:
# Prepare data for warna classification
df_warna = train_df[['image_path', 'warna']].copy()
df_warna.columns = ['image', 'label']

# Split train/validation
train_warna, val_warna = train_test_split(
    df_warna, test_size=0.2, random_state=42, stratify=df_warna['label']
)

print(f"Training samples: {len(train_warna)}")
print(f"Validation samples: {len(val_warna)}")

# Initialize AutoGluon predictor
predictor_warna = MultiModalPredictor(
    label='label',
    path='automl_warna_model'
)

# Let AutoGluon decide everything
print("\nTraining WARNA model - AutoGluon will automatically select best pipeline...")
predictor_warna.fit(
    train_data=train_warna,
    time_limit=600  # 10 minutes
)

print("\nWARNA model training completed")

## 8. Inspect WARNA Model Pipeline (Transparency)

In [ ]:
print("=" * 70)
print("WARNA MODEL PIPELINE TRANSPARENCY")
print("=" * 70)

print("\n1. MODEL ARCHITECTURE:")
print(f"   Model path: {predictor_warna.path}")
print(f"   Problem type: {predictor_warna.problem_type}")
print(f"   Number of classes: 5 (merah, kuning, biru, hitam, putih)")

print("\n2. SAVED ARTIFACTS:")
model_files = os.listdir(predictor_warna.path)
for f in sorted(model_files)[:10]:
    print(f"   - {f}")

print("\n3. MODEL SUMMARY:")
try:
    info = predictor_warna.info()
    for key, value in info.items():
        print(f"   {key}: {value}")
except:
    print("   Model info available after predictions")

print("\n" + "=" * 70)

## 9. Evaluate WARNA Model

In [ ]:
# Evaluate on validation set
val_score_warna = predictor_warna.evaluate(val_warna)
print("WARNA Model Performance:")
print(f"Validation Accuracy: {val_score_warna['accuracy']:.4f}")

# Get predictions
predictions_warna = predictor_warna.predict(val_warna)
print(f"\nSample predictions: {predictions_warna[:10].tolist()}")
print(f"Sample true labels: {val_warna['label'].values[:10].tolist()}")

## 10. Performance Summary

In [ ]:
summary = pd.DataFrame({
    'Model': ['JENIS (Type)', 'WARNA (Color)'],
    'Classes': [2, 5],
    'Train Samples': [len(train_jenis), len(train_warna)],
    'Val Samples': [len(val_jenis), len(val_warna)],
    'Accuracy': [val_score['accuracy'], val_score_warna['accuracy']]
})

print("=" * 70)
print("AUTOML PERFORMANCE SUMMARY")
print("=" * 70)
print(summary.to_string(index=False))
print("=" * 70)
print("\nBoth models trained with fully automated pipeline")
print("AutoGluon automatically selected optimal:")
print("- Model architecture")
print("- Hyperparameters")
print("- Training strategies")
print("- Data augmentation")

## 11. Predict on Test Set

In [ ]:
# Get test image IDs
test_dir = 'test/test'
test_ids = []
test_paths = []

for filename in sorted(os.listdir(test_dir)):
    if filename.endswith(('.jpg', '.png')):
        img_id = int(filename.split('.')[0])
        test_ids.append(img_id)
        test_paths.append(os.path.join(test_dir, filename))

print(f"Found {len(test_ids)} test images")

# Prepare test dataframe
test_df = pd.DataFrame({
    'image': test_paths
})

print("\nPredicting JENIS (Type)...")
pred_jenis = predictor_jenis.predict(test_df)

print("Predicting WARNA (Color)...")
pred_warna = predictor_warna.predict(test_df)

print("\nPredictions completed")

## 12. Create Submission File

In [ ]:
# Create submission dataframe
submission = pd.DataFrame({
    'id': test_ids,
    'jenis': pred_jenis,
    'warna': pred_warna
})

# Sort by id
submission = submission.sort_values('id').reset_index(drop=True)

# Save to CSV
submission.to_csv('submission_autogluon.csv', index=False)

print("Submission file created: submission_autogluon.csv")
print(f"\nFirst 10 predictions:")
print(submission.head(10))
print(f"\nTotal predictions: {len(submission)}")

# Show prediction distribution
print("\nPrediction distribution:")
print(f"JENIS - Kaos: {(submission['jenis']==0).sum()}, Hoodie: {(submission['jenis']==1).sum()}")
print(f"WARNA - Merah: {(submission['warna']==0).sum()}, Kuning: {(submission['warna']==1).sum()}, Biru: {(submission['warna']==2).sum()}, Hitam: {(submission['warna']==3).sum()}, Putih: {(submission['warna']==4).sum()}")